<center><h1>Xie_Chong_HW7</h1></center>

Name: Chong Xie
<br>
Github Username: cxieusc
<br>
USC ID: 6800668300

## 1. Multi-class and Multi-Label Classification Using Support Vector Machines

Import packages

In [65]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

import warnings
warnings.filterwarnings("ignore")

### (a) Download the Anuran Calls (MFCCs) Data Set

In [49]:
df = pd.read_csv('../data/Anuran Calls (MFCCs)/Frogs_MFCCs.csv')
df.head(3)

,MFCCs_ 1,MFCCs_ 2,MFCCs_ 3,MFCCs_ 4,MFCCs_ 5,MFCCs_ 6,MFCCs_ 7,MFCCs_ 8,MFCCs_ 9,MFCCs_10,...,MFCCs_17,MFCCs_18,MFCCs_19,MFCCs_20,MFCCs_21,MFCCs_22,Family,Genus,Species,RecordID
0,1.0,0.152936,-0.105586,0.200722,0.317201,0.260764,0.100945,-0.150063,-0.171128,0.124676,...,-0.108351,-0.077623,-0.009568,0.057684,0.118680,0.014038,Leptodactylidae,Adenomera,AdenomeraAndre,1
1,1.0,0.171534,-0.098975,0.268425,0.338672,0.268353,0.060835,-0.222475,-0.207693,0.170883,...,-0.090974,-0.056510,-0.035303,0.020140,0.082263,0.029056,Leptodactylidae,Adenomera,AdenomeraAndre,1
2,1.0,0.152317,-0.082973,0.287128,0.276014,0.189867,0.008714,-0.242234,-0.219153,0.232538,...,-0.050691,-0.023590,-0.066722,-0.025083,0.099108,0.077162,Leptodactylidae,Adenomera,AdenomeraAndre,1


In [50]:
df = df.drop(columns=['RecordID'])
df_train, df_test = train_test_split(df, train_size=0.7)

In [59]:
print(f"df_train shape: {df_train.shape}")
print(f"df_test shape: {df_test.shape}")

df_train shape: (5036, 25)
df_test shape: (2159, 25)


### (b) Train a classifier for each label

#### (i) Research

**Exact Match**

Exact match measures the proportion of samples for which all three labels are predicted correctly. If any label is incorrect, the entire sample is considered incorrect. Higher values indicate better performance.

$$
\text{Exact Match}
=
\frac{1}{N}\sum_{i=1}^{N}
\mathbf{1}\left(y_{ij}=\hat{y}_{ij}\text{ for all }j\right)
$$

Reference: [scikit-learn — accuracy_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html)

**Hamming Loss**

Hamming loss measures the proportion of incorrectly predicted label positions. In this problem, errors are averaged over all samples and the three categorical outputs. Lower values indicate better performance.

$$
\text{Hamming Loss}
=
\frac{1}{NL}\sum_{i=1}^{N}\sum_{j=1}^{L}
\mathbf{1}(y_{ij}\neq\hat{y}_{ij})
$$


Reference: [scikit-learn — hamming_loss](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.hamming_loss.html)

**Hamming Score**

Hamming score is defined as the proportion of correctly predicted label positions. Higher values indicate better performance.

$$
\text{Hamming Score}
=
1-\text{Hamming Loss}
=
\frac{1}{NL}\sum_{i=1}^{N}\sum_{j=1}^{L}
\mathbf{1}(y_{ij}=\hat{y}_{ij})
$$

**We will use all the methods above to evaluate all the models**

#### (ii) Train a SVM for each of the labels

In [52]:
label_cols = ['Family', 'Genus', 'Species']

X_train = df_train.drop(columns=label_cols)
y_train = df_train[label_cols]
X_test = df_test.drop(columns=label_cols)
y_test = df_test[label_cols]

In [53]:
# Initial search ranges for C and gamma
C_values = [0.1, 1, 10, 100, 1000]
sigma_values = np.linspace(0.1, 2.0, 20)
param_grid = {
    "estimator__C": C_values,
    "estimator__gamma": 1 / (2 * sigma_values ** 2)
}

rbf_models = {}
rbf_results = []

y_pred_rbf = pd.DataFrame(index=y_test.index)

for label in label_cols:
    print(f"Training classifier for {label}...")

    model = OneVsRestClassifier(
        SVC(kernel="rbf")
    )

    search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        cv=10,
        n_jobs=-1,
    )

    search.fit(X_train, y_train[label])

    rbf_models[label] = search.best_estimator_
    y_pred_rbf[label] = search.predict(X_test)

    rbf_results.append({
        "Label": label,
        "Best C": search.best_params_["estimator__C"],
        "Best gamma": search.best_params_["estimator__gamma"],
        "CV Accuracy": search.best_score_,
        "Test Accuracy": (
            y_pred_rbf[label] == y_test[label]
        ).mean(),
        "Best sigma": np.sqrt(
            1 / (2 * search.best_params_["estimator__gamma"])
        ),
    })

rbf_results = pd.DataFrame(rbf_results)
display(rbf_results)

Training classifier for Family...
Training classifier for Genus...
Training classifier for Species...


,Label,Best C,Best gamma,CV Accuracy,Test Accuracy,Best sigma
0,Family,10,2.000000,0.994042,0.993052,0.5
1,Genus,100,2.000000,0.990666,0.989810,0.5
2,Species,10,1.388889,0.991260,0.990736,0.6


In [54]:
correct = (
    y_test[label_cols].to_numpy()
    == y_pred_rbf[label_cols].to_numpy()
)

rbf_metrics = pd.DataFrame([{
    "Model": "RBF SVM",
    "Exact Match": correct.all(axis=1).mean(),
    "Hamming Score": correct.mean(),
    "Hamming Loss": (~correct).mean()
}])

display(rbf_metrics)

,Model,Exact Match,Hamming Score,Hamming Loss
0,RBF SVM,0.988421,0.9912,0.0088


#### (iii) Repeat 1(b)ii with L1-penalized SVMs

In [55]:
param_grid_l1 = {
    "svm__C": [0.001, 0.01, 0.1, 1, 10, 100, 1000]
}

l1_models = {}
l1_results = []
y_pred_l1 = pd.DataFrame(index=y_test.index)

for label in label_cols:
    print(f"Training L1 SVM for {label}...")

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("svm", LinearSVC(
            penalty="l1",
            loss="squared_hinge",
            dual=False,
            max_iter=50000
        ))
    ])

    search = GridSearchCV(
        estimator=model,
        param_grid=param_grid_l1,
        scoring="accuracy",
        cv=10,
        n_jobs=-1
    )

    search.fit(X_train, y_train[label])

    l1_models[label] = search.best_estimator_
    y_pred_l1[label] = search.predict(X_test)

    l1_results.append({
        "Label": label,
        "Best C": search.best_params_["svm__C"],
        "CV Accuracy": search.best_score_,
        "Test Accuracy": (
            y_pred_l1[label] == y_test[label]
        ).mean()
    })

l1_results = pd.DataFrame(l1_results)
display(l1_results)

Training L1 SVM for Family...
Training L1 SVM for Genus...
Training L1 SVM for Species...


,Label,Best C,CV Accuracy,Test Accuracy
0,Family,10,0.932090,0.945345
1,Genus,10,0.949961,0.960167
2,Species,10,0.956711,0.968041


In [56]:
correct_l1 = (
    y_test[label_cols].to_numpy()
    == y_pred_l1[label_cols].to_numpy()
)

l1_metrics = pd.DataFrame([{
    "Model": "L1 Linear SVM",
    "Exact Match": correct_l1.all(axis=1).mean(),
    "Hamming Score": correct_l1.mean(),
    "Hamming Loss": (~correct_l1).mean()
}])

display(l1_metrics)

,Model,Exact Match,Hamming Score,Hamming Loss
0,L1 Linear SVM,0.928208,0.957851,0.042149


#### (iv) Repeat 1(b)iii by using SMOTE or any other method for imbalance

In [57]:
param_grid_smote = {
    "svm__C": [0.001, 0.01, 0.1, 1, 10, 100]
}

smote_models = {}
smote_results = []
y_pred_smote = pd.DataFrame(index=y_test.index)

for label in label_cols:
    print(f"Training L1 SVM with SMOTE for {label}...")

    model = ImbPipeline([
        ("scaler", StandardScaler()),
        ("smote", SMOTE()),
        ("svm", LinearSVC(
            penalty="l1",
            loss="squared_hinge",
            dual=False,
            max_iter=50000
        ))
    ])

    search = GridSearchCV(
        estimator=model,
        param_grid=param_grid_smote,
        scoring="accuracy",
        cv=10,
        n_jobs=-1
    )

    search.fit(X_train, y_train[label])

    smote_models[label] = search.best_estimator_
    y_pred_smote[label] = search.predict(X_test)

    smote_results.append({
        "Label": label,
        "Best C": search.best_params_["svm__C"],
        "CV Accuracy": search.best_score_,
        "Test Accuracy": (
            y_pred_smote[label] == y_test[label]
        ).mean()
    })

smote_results = pd.DataFrame(smote_results)
display(smote_results)

Training L1 SVM with SMOTE for Family...
Training L1 SVM with SMOTE for Genus...
Training L1 SVM with SMOTE for Species...


,Label,Best C,CV Accuracy,Test Accuracy
0,Family,10,0.917004,0.918944
1,Genus,10,0.909652,0.918944
2,Species,10,0.955126,0.963409


In [58]:
correct_smote = (
    y_test[label_cols].to_numpy()
    == y_pred_smote[label_cols].to_numpy()
)

smote_metrics = pd.DataFrame([{
    "Model": "L1 Linear SVM + SMOTE",
    "Exact Match": correct_smote.all(axis=1).mean(),
    "Hamming Score": correct_smote.mean(),
    "Hamming Loss": (~correct_smote).mean()
}])

comparison = pd.concat(
    [rbf_metrics, l1_metrics, smote_metrics],
    ignore_index=True
)

display(comparison)

,Model,Exact Match,Hamming Score,Hamming Loss
0,RBF SVM,0.988421,0.991200,0.008800
1,L1 Linear SVM,0.928208,0.957851,0.042149
2,L1 Linear SVM + SMOTE,0.868921,0.933766,0.066234


The RBF SVM achieved the best test performance, with an exact match
of 0.988421, a Hamming score of 0.991200, and a Hamming loss of 0.008800.

Adding SMOTE to the L1-penalized linear SVM reduced exact match from
0.928208 to 0.868921 and Hamming score from 0.957851 to 0.933766.
Hamming loss increased from 0.042149 to 0.066234.

Therefore, SMOTE did not improve overall test performance under these
metrics in this experiment. 

## 2. K-Means Clustering on a Multi-Class and Multi-Label Data Set

### (a) Use k-means clustering

In [67]:
X = df.drop(columns=label_cols)
# k needs to be at least 2 for Silhouette score
k_lst = [k for k in range(2, 51)]
max_silhouette_score = -1
best_k = None

for k in k_lst:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X)
    score = silhouette_score(X, kmeans.labels_)
    if score > max_silhouette_score:
        max_silhouette_score = score
        best_k = k

print(f"Best k: {best_k}, Best Silhouette Score: {max_silhouette_score}")

Best k: 4, Best Silhouette Score: 0.3787509343305295


In [68]:
km = KMeans(n_clusters=best_k, random_state=42)
y_pred = km.fit_predict(X)

### (b) Determine which family is the majority

In [69]:
cluster_labels = df[label_cols].copy()
cluster_labels["Cluster"] = y_pred

# Find the majority class for each label within each cluster
majority_labels = (
    cluster_labels
    .groupby("Cluster")[label_cols]
    .agg(lambda col: col.mode().iloc[0])
)

display(majority_labels)

,Family,Genus,Species
Cluster,,,
0,Leptodactylidae,Adenomera,AdenomeraHylaedactylus
1,Hylidae,Hypsiboas,HypsiboasCinerascens
2,Dendrobatidae,Ameerega,Ameeregatrivittata
3,Hylidae,Hypsiboas,HypsiboasCordobae


### (c) Calculate the average Hamming distance, Hamming score, and Hamming loss

In [70]:
cluster_predictions = majority_labels.loc[y_pred, label_cols].to_numpy()
true_labels = df[label_cols].to_numpy()

# Compare Family, Genus, and Species separately
incorrect = true_labels != cluster_predictions

avg_hamming_distance = incorrect.sum(axis=1).mean()
hamming_loss = incorrect.mean()
hamming_score = 1 - hamming_loss

clustering_metrics = pd.DataFrame([{
    "Average Hamming Distance": avg_hamming_distance,
    "Hamming Score": hamming_score,
    "Hamming Loss": hamming_loss
}])

display(clustering_metrics)

,Average Hamming Distance,Hamming Score,Hamming Loss
0,0.667269,0.777577,0.222423


### Monte-Carlo Simulation

In [73]:
X = df.drop(columns=label_cols)
true_labels = df[label_cols].to_numpy()

mc_results = []

for run in range(50):
    # (a) Select k using the silhouette score
    best_k = None
    best_score = -np.inf
    best_clusters = None

    for k in range(2, 51):
        kmeans = KMeans(
            n_clusters=k,
            random_state=run,
            n_init=10
        )
        clusters = kmeans.fit_predict(X)
        score = silhouette_score(X, clusters)

        if score > best_score:
            best_score = score
            best_k = k
            best_clusters = clusters.copy()

    # (b) Find the majority label for each output in each cluster
    labels_with_clusters = df[label_cols].copy()
    labels_with_clusters["Cluster"] = best_clusters

    majority = (
        labels_with_clusters
        .groupby("Cluster")[label_cols]
        .agg(lambda col: col.mode().iloc[0])
    )

    # (c) Assign cluster labels and calculate metrics
    predictions = majority.loc[
        best_clusters, label_cols
    ].to_numpy()

    incorrect = true_labels != predictions

    distance = incorrect.sum(axis=1).mean()
    loss = incorrect.mean()
    score = 1 - loss

    mc_results.append({
        "Run": run + 1,
        "Best k": best_k,
        "Silhouette": best_score,
        "Average Hamming Distance": distance,
        "Hamming Score": score,
        "Hamming Loss": loss
    })

    if (run + 1) % 10 == 0:
        recent_avg = np.mean([
            result["Average Hamming Distance"]
            for result in mc_results[-10:]
        ])

        print(
            f"Runs {run - 8}-{run + 1}: "
            f"Average Hamming Distance = {recent_avg:.6f}",
            flush=True
        )

mc_results = pd.DataFrame(mc_results)
display(mc_results)

Runs 1-10: Average Hamming Distance = 0.674079
Runs 11-20: Average Hamming Distance = 0.666977
Runs 21-30: Average Hamming Distance = 0.677331
Runs 31-40: Average Hamming Distance = 0.673871
Runs 41-50: Average Hamming Distance = 0.667186


,Run,Best k,Silhouette,Average Hamming Distance,Hamming Score,Hamming Loss
0,1,4,0.378751,0.667269,0.777577,0.222423
1,2,4,0.378751,0.667269,0.777577,0.222423
2,3,4,0.385256,0.735372,0.754876,0.245124
3,4,4,0.378751,0.667269,0.777577,0.222423
4,5,4,0.378751,0.667269,0.777577,0.222423
5,6,4,0.378751,0.667269,0.777577,0.222423
6,7,4,0.378751,0.667269,0.777577,0.222423
7,8,4,0.378751,0.667269,0.777577,0.222423
8,9,4,0.378751,0.667269,0.777577,0.222423
9,10,4,0.378751,0.667269,0.777577,0.222423


In [74]:
avg_dist = mc_results["Average Hamming Distance"].mean()
std_dist = mc_results["Average Hamming Distance"].std(ddof=1)

print(f"Average Hamming Distance: {avg_dist:.6f}")
print(f"Standard Deviation: {std_dist:.6f}")

Average Hamming Distance: 0.671889
Standard Deviation: 0.016978


## 3. ISLR 12.6.2